In [147]:
# Starting the omlx server
!/opt/homebrew/bin/omlx start

oMLX server running on port 8000


In [148]:
#import statements and keys


import os
from pathlib import Path
from dotenv import load_dotenv
import os
import yaml
import logging
import re
from datetime import datetime
import ast
import yaml
import time
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
env_path = Path.cwd().parent / "environment.env"

# Load the .env file
if load_dotenv(dotenv_path=env_path):
    print("Environment variables loaded successfully!")
else:
    print("Error: Could not find or load the .env file.")

OmlxAPI = os.getenv('OmlxAPI')
baseurl = os.getenv('baseurl')


Environment variables loaded successfully!


In [149]:
def llm_response_qwen(prompt: str, ) -> str:
    """Generic function to get a response from the Groq Qwen LLM."""
    try:
        client = OpenAI(
            base_url=baseurl,  # Your oMLX local address
            api_key=OmlxAPI          # Your oMLX API Key
        )
        response = client.chat.completions.create(
            model="mlx-community/gemma-4-12B-it-OptiQ-4bit",           # Replace with your loaded model ID
            messages=[{"role": "user", "content": prompt}]
        )
             
        content = response.choices[0].message.content
        return content
    except Exception as e:
        logging.error(f"An error occurred while communicating with the Groq API: {e}")
        return ""

In [150]:
def yamlcheck(content):
    try:
        raw_response = content.strip()
    except:
        None

    # Remove markdown code fences if Qwen accidentally includes them
    if raw_response.startswith("```"):
        raw_response = (raw_response.replace("```python", "").replace("```", "").strip()
        )

    try:
        # Safely convert the string representation of a list into a real Python list
        content = ast.literal_eval(raw_response)
    except Exception as e:
        print(f"Error parsing list: {e}")
        # Fallback: just split by lines if it completely fails
        content = [line.strip() for line in raw_response.split("\n") if line.strip()]
        
    return content

In [151]:
jobdescription = input('Enter the job description')

In [152]:
#Load the content files

path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
with open(path, 'r') as f:
    resume = yaml.safe_load(f)
# a = list(resume['Professional Experience'].keys())

#Load Destination Files
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)



#Create loop to get tailored initial responses with checking all the experiences are available

if (len(resume["work_experience"]) ==len(test['work_experience'])):
    for i in range(len(resume["work_experience"])):
    # for i in range(1):
        experience = resume["work_experience"][i]["achievements"]
        prompt1 = f"""[Task] rewrite the points to suit the job description and resume and sort based on importance to the job and select top 7 

        [Constraints]
        - Truthfulness: Use ONLY metrics, tools, and software explicitly in the Resume. Never add/change tools or numbers.
        - Format: Return ONLY a raw, valid Python list of 7 strings on ONE continuous line. No markdown blocks (```), no \n, no indents. Start with [ and end with ].
        - Content: Blend problem, solution, and impact seamlessly. No labels like "Problem:" or "Impact:".
        
        NO PREAMBLE

        [Data]
        Resume: {experience}
        Job: {jobdescription}

        [Guardrail] Ensure exactly 7 elements are in the list. Zero invented skills/tools."""

    
        tailoredresponse = llm_response_qwen(prompt1)
        cleanedresponse = yamlcheck(tailoredresponse)

        test['work_experience'][i]['achievements'] = cleanedresponse

    # Now saving to YAML will be perfectly clean and structured
    with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

    print("inital tailored resume collected")
else:
    print("There is not same number of experiences in source and destination")

2026-07-13 23:09:23,596 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-13 23:09:38,770 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-13 23:09:53,425 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-13 23:10:07,261 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


inital tailored resume collected


In [153]:
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)

output = [{"name": "Core Competencies", "keywords": [ "Strategic Procurement", "Supplier Negotiation", "Procurement Analytics"]},{"name": "Core Competencies", "keywords": [ "Strategic Procurement", "Supplier Negotiation", "Procurement Analytics"]}]


skillsprompt = f""" Rewrite the skills section in a crisp one or two word to suit the below tailored resume and job description maintain same yaml format can modify or create new skill sections and rate the final skills plus reume for ats compatablity to the job description.
- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

skills: {test['skills']}
Job: {jobdescription}
resume: {test['work_experience']}

Example:
OUTPUT : {output}

NO PREAMBLE
"""


tailoredresponse = llm_response_qwen(skillsprompt)
cleanedresponse = yamlcheck(tailoredresponse)
test['skills'] = cleanedresponse[0]['skills']
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

2026-07-13 23:10:21,019 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [154]:
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)
output = {"summary":"Strategic Procurement and Sourcing Professional with over 7 years of experience in "}

summaryprompt = f"""rewrite the summary section to suit the job description and resume
- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

Output : {output}

summary: {test['summary']}
Job: {jobdescription}
resume: {test['work_experience']}



NO PREAMBLE
"""

tailoredresponse = llm_response_qwen(summaryprompt)
cleanedresponse = yamlcheck(tailoredresponse)
try:
    cleanedresponse[0]['summary']
except:
      test['summary'] = cleanedresponse['summary']
      
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)


2026-07-13 23:10:51,824 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [155]:
# cleanedresponse

In [156]:
# cleanedresponse = yamlcheck(tailoredresponse)
# try:
#     cleanedresponse[0]['summary']
# except:
#       test['summary'] = cleanedresponse['summary']
      
# with open("output.yaml", "w", encoding="utf-8") as f:
#         yaml.dump(test, f, sort_keys=False, default_flow_style=False)

In [157]:
path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)
    
output = {"project_name": "Procurement Analytics Dashboard",
    "description": "Developed a Power BI dashboard to track procurement KPIs, supplier performance, and category spend, providing actionable insights for informed purchasing decisions.",
    "keywords" : ["Power BI", "Procurement", "Supply Chain", "KPIs"]}

projectprompt = f"""Select/ Rewrite 4 projects from the below list to suit the job description

- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

Output : {output}

projects: {resume['projects']}
Job: {jobdescription}

NO PREAMBLE

"""

tailoredresponse = llm_response_qwen(projectprompt)
cleanedresponse = yamlcheck(tailoredresponse)

test['projects']= cleanedresponse
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

2026-07-13 23:11:07,294 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [172]:
#make the pdf
!python ReportLabs.py 

✅ Resume generated: Karthikeyan_Baskaran_Resume.pdf


In [159]:
# import yaml
# path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
# with open(path, 'r') as f:
#     test = yaml.safe_load(f)

In [160]:
# prompt = f""" match between the resume and the job give me a score out of 10
#     Resume: {test}
#     Job: {jobdescription}
# """

In [161]:
# prompt

<!-- prompt -->

In [162]:
# from openai import OpenAI

# client = OpenAI(
#     base_url="http://127.0.0.1:8000/v1",  # Your oMLX local address
#     api_key="1234"          # Your oMLX API Key
# )

# response = client.chat.completions.create(
#     model="mlx-community/gemma-4-12B-it-OptiQ-4bit",           # Replace with your loaded model ID
#     messages=[{"role": "user", "content": prompt}]
# )

# print(response.choices[0].message.content)

In [163]:
# type(response)

In [164]:
# # 1. Extract the raw string from the object
# raw_content = response.choices[0].message.content

# print(raw_content)

In [165]:
# raw_content

In [166]:
# content = ""
# for chunk in response:
#     print(chunk)
#     break

#     # c = chunk.choices[0].delta.content if chunk.choices[0].delta.content else ""
#     # content += c

In [167]:
# path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
# with open(path, 'r') as f:
#     resume = yaml.safe_load(f)

In [168]:
# resume

In [169]:
# answer = ["Automated supplier performance monitoring and provided early warnings for dependency risks by building custom SQL-based tracking systems, cutting manual reporting time by 50%.", "Developed procurement and contract documents by designing interactive Power BI dashboards that transformed messy procurement data into actionable insights to speed up sourcing workflows.", "Maintained a 'single source of truth' for end-to-end supply chain visibility and data accuracy within Sage ERP to support leadership decision-making.", "Streamlined data mining and reporting processes by engineering complex queries to transform raw numbers into monthly performance reports.", "Reduced stockouts and achieved 98% production uptime by re-engineering purchase planning for global suppliers with long lead times.", "Spearheaded strategic sourcing initiatives that slashed procurement costs while diversifying the supplier base to lower organizational risk.", "Improved vendor accountability and lead-time reliability through a high-engagement relationship model, resulting in a 30% improvement in on-time deliveries."]

In [170]:
# answer